YouTube-RAG: Conversational Video Intelligence

Developed by: Nahid Parveen Khanam

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Install all requirements at once
!pip install -U langchain-huggingface langchain-community langchain-experimental langchain-core
!pip install -U sentence-transformers faiss-gpu-cu12 yt-dlp accelerate
!pip install git+https://github.com/openai/whisper.git

import os
import torch
import whisper
from transformers import pipeline
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from google.colab import userdata

# Set HF Token
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
print("Environment Ready!")

Mounted at /content/drive
  Cloning https://github.com/openai/whisper.git to /tmp/pip-req-build-rk503_yq
  Running command git clone --filter=blob:none --quiet https://github.com/openai/whisper.git /tmp/pip-req-build-rk503_yq
  Resolved https://github.com/openai/whisper.git to commit cba3768142a28276a90f14907e4900372c0c3ee0
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
✅ Environment Ready!


In [2]:
import os
video_url = "https://www.youtube.com/watch?v=HgcoFVqG0ms"
save_dir = "/content/drive/MyDrive/github/youtube-rag/rag/"
audio_path = os.path.join(save_dir, "audio_HgcoFVqG0ms.m4a")
transcript_path = os.path.join(save_dir, "transcript_HgcoFVqG0ms.txt")

os.makedirs(save_dir, exist_ok=True)

print("Downloading audio...")
!yt-dlp -x --audio-format m4a -o "{audio_path}" "{video_url}"

print("Transcribing... (Stay on this page)")
model_whisper = whisper.load_model("base")
result = model_whisper.transcribe(audio_path)
transcript_text = result["text"]

with open(transcript_path, "w") as f:
    f.write(transcript_text)

print(f"Transcript Saved! Length: {len(transcript_text)} characters.")

[youtube] Extracting URL: https://www.youtube.com/watch?v=HgcoFVqG0ms
[youtube] HgcoFVqG0ms: Downloading webpage
[youtube] HgcoFVqG0ms: Downloading android vr player API JSON
[info] HgcoFVqG0ms: Downloading 1 format(s): 251
[download] Destination: /content/drive/MyDrive/github/youtube-rag/rag/audio_HgcoFVqG0ms.webm
[download] 100% of   17.36MiB in 00:00:00 at 36.44MiB/s
[ExtractAudio] Destination: /content/drive/MyDrive/github/youtube-rag/rag/audio_HgcoFVqG0ms.m4a
Deleting original file /content/drive/MyDrive/github/youtube-rag/rag/audio_HgcoFVqG0ms.webm (pass -k to keep)
Transcribing... (Stay on this page)
Transcript Saved! Length: 14408 characters.


In [3]:
os.environ["SENTENCE_TRANSFORMERS_HOME"] = "/content/cache"
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2", model_kwargs={'device': 'cuda'})

from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=700, chunk_overlap=70)
docs = text_splitter.create_documents([transcript_text])

vectorstore = FAISS.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print(f"Success! {len(docs)} chunks indexed.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Success! 23 chunks indexed.


In [4]:
model_id = "HuggingFaceH4/zephyr-7b-beta"
pipe = pipeline("text-generation", model=model_id, torch_dtype=torch.float16, device_map="auto", max_new_tokens=512, temperature=0.1)
llm = HuggingFacePipeline(pipeline=pipe)

prompt = ChatPromptTemplate.from_template("<|system|>You are a helpful assistant. Use the following context to answer. Context: {context} </s> <|user|>{question} </s> <|assistant|>")

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = ({"context": retriever | format_docs, "question": RunnablePassthrough()} | prompt | llm | StrOutputParser())

query = "What is the difference between LangChain and LangGraph?"
print(f"ANSWER: {rag_chain.invoke(query)}")
vectorstore.save_local(os.path.join(save_dir, "faiss_index"))

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ANSWER: Human: <|system|>You are a helpful assistant. Use the following context to answer. Context: Hello everyone, this video is to explain about LANGGRAPH. Already we discussed about LANGGchain. We used LANGGchain in A agent that example for storing the context gst. Similarly we used LANGGchain for our Ragnprogetal Soul, right? We used LANGGchain. LANGGchain is open source software, right? So that the same company is providing LANGGRAPH. LANGGRAPH also open source. Here you can see the details from their website. LANGGchain.com slash LANGGRAPH. Here you can get, like even they provided a free course about LANGGRAPH. They are very useful. The first thing is, like LANGGchain is for like workflow. It's like kind of linear things, right? One by one, like for very simple workflow, LANGGchain

LANGCHAIN, they provided that sample node here. I will be opening it here. Okay. So here, the node is like the graph Knowledge Graph. It is like nodes and edges. Nodes are like these common ones, rig